In [1]:
import pandas as pd 

df = pd.read_csv('../wikidata_global_companies_info.csv') 
df.head(2)

,Unnamed: 0,wikidata_uri,company name,description,country,instance of,inception,official website,industry,founded by
0,0,http://www.wikidata.org/entity/Q279260,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",Austria,['hotel' 'Viennese coffee house'],1876-01-01,http://www.sacher.com/,['hotel'],Édouard Sacher
1,1,http://www.wikidata.org/entity/Q193326,Goldman Sachs,American investment bank,United States of America,['stock exchange' 'bank' 'multinational corpor...,1869-01-01,https://www.goldmansachs.com/,['financial services' 'financial sector'\n 'fi...,Samuel Sachs


In [2]:
df2 = pd.read_csv('../classification-dataset-v1.csv') 
df2.head(2)

,Category,website,company_name,homepage_text,h1,h2,h3,nav_link_text,meta_keywords,meta_description
0,Commercial Services & Supplies,bipelectric.com,bip dipietro electric inc,Electrici...,NaN,NaN,NaN,NaN,"electricians vero beach, vero beach electrical...","Providing quality, reliable full service resid..."
1,Healthcare,eliasmedical.com,elias medical,site map | en español Elias Medical h...,Offering Bakersfield family medical care from ...,Welcome to ELIAS MEDICAL#sep#Family Medical Pr...,Get To Know Elias Medical#sep#Family Medical P...,NaN,Elias Medical bakersfield ca family doctor med...,For the best value in Bakersfield skin care tr...


In [3]:
df2 = df2.rename(columns = {'company_name': 'company name' , 'meta_description' : 'description' , 'Category': 'industry' })

In [4]:
df = df[['company name' , 'description' , 'industry']] 
df2 = df2[['company name' , 'description' , 'industry']]  

In [5]:
data = pd.concat([df, df2]) 
data.head() 

,company name,description,industry
0,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",['hotel']
1,Goldman Sachs,American investment bank,['financial services' 'financial sector'\n 'fi...
2,Sberbank,Russian banking and financial services company,['banking in Russia' 'financial sector'\n 'fin...
3,Heineken,Dutch beer company,['beverage industry']
4,Deutsche Bank,German global banking and financial services c...,['financial services' 'Other monetary intermed...


In [6]:
data.loc[ data['company name'] == 'Omni Social App' , 'industry'  ] =  'technology'
data.loc[ data['company name'] == 'Toni & Guy' , 'industry'  ] =  'hairdresser'


In [7]:

print(data[data['company name'] == 'Omni Social App'])
print(data[data['company name'] == 'Toni & Guy'])

         company name         description    industry
2663  Omni Social App  Technology company  technology
    company name       description     industry
468   Toni & Guy  hair salon chain  hairdresser


In [8]:
data['original_industry'] = data['industry']
data.head() 

,company name,description,industry,original_industry
0,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",['hotel'],['hotel']
1,Goldman Sachs,American investment bank,['financial services' 'financial sector'\n 'fi...,['financial services' 'financial sector'\n 'fi...
2,Sberbank,Russian banking and financial services company,['banking in Russia' 'financial sector'\n 'fin...,['banking in Russia' 'financial sector'\n 'fin...
3,Heineken,Dutch beer company,['beverage industry'],['beverage industry']
4,Deutsche Bank,German global banking and financial services c...,['financial services' 'Other monetary intermed...,['financial services' 'Other monetary intermed...


In [9]:
print(data.iloc[0, 2])
print(type(data.iloc[0, 2]))

['hotel']
<class 'str'>


In [10]:
data.shape

(77553, 4)

In [11]:
original = pd.read_csv('../ExioNAICS.csv') 
original = original[['NAICS_2 Title' , 'NAICS_2 Code']] 

SECTOR_MERGE = {
    '31': '31-33', '32': '31-33', '33': '31-33',
    '44': '44-45', '45': '44-45',
    '48': '48-49', '49': '48-49',
} 
original['NAICS_2 Code'] = original['NAICS_2 Code'].astype(str)
original['NAICS_2 Code'] = original['NAICS_2 Code'].map(SECTOR_MERGE).fillna(original['NAICS_2 Code'])

In [12]:
title_code_matching = {}

titles = original['NAICS_2 Title' ].to_list() 
codes = original['NAICS_2 Code'].to_list()

for i in range(len(titles)):
    if titles[i] not in title_code_matching:
        title_code_matching[titles[i]] = codes[i] 

title_code_matching

{'Retail Trade': '44-45',
 'Information': '51',
 'Manufacturing': '31-33',
 'Finance and Insurance': '52',
 'Mining, Quarrying, and Oil and Gas Extraction': '21',
 'Other Services (except Public Administration)': '81',
 'Wholesale Trade': '42',
 'Public Administration': '92',
 'Agriculture, Forestry, Fishing and Hunting': '11',
 'Transportation and Warehousing': '48-49',
 'Real Estate and Rental and Leasing': '53',
 'Professional, Scientific, and Technical Services': '54',
 'Construction': '23',
 'Arts, Entertainment, and Recreation': '71',
 'Health Care and Social Assistance': '62',
 'Administrative and Support and Waste Management and Remediation Services': '56',
 'Accommodation and Food Services': '72',
 'Utilities': '22',
 'Educational Services': '61',
 'Management of Companies and Enterprises': '55'}

In [13]:
data['industry']  = data['industry'].str.lower()
data['industry'] = data['industry'].str.replace(r"[\[\]'\"\n,]" , '' , regex = True)
data['industry']  = data['industry'].str.split(" ")

In [14]:
# The complete updated synonym dictionary
synonyms = {
    # --- Food, Stay & Travel -> Matches "Accommodation and Food Services" ---
    'hotel': 'accommodation',
    'motel': 'accommodation',
    'hospitality': 'accommodation',
    'restaurant': 'accommodation',
    'gastronomy': 'accommodation',
    'foodservice': 'accommodation',
    'tourism': 'accommodation',
    'horeca': 'accommodation',
    'coffee': 'accommodation',
    'coffeehouse': 'accommodation',

    # --- Making Things -> Matches "Manufacturing" ---
    'brewing': 'manufacturing',
    'brewery': 'manufacturing',
    'beverage': 'manufacturing',
    'beer': 'manufacturing',
    'automotive': 'manufacturing',
    'vehicle': 'manufacturing',
    'clothing': 'manufacturing',
    'fashion': 'manufacturing',
    'textile': 'manufacturing',
    'footwear': 'manufacturing',
    'cosmetics': 'manufacturing',
    'pharmaceutical': 'manufacturing',
    'biotechnology': 'manufacturing',
    'biopharmaceutical': 'manufacturing',
    'chemical': 'manufacturing',
    'electronics': 'manufacturing',
    'semiconductor': 'manufacturing',
    'hardware': 'manufacturing',
    'aerospace': 'manufacturing',
    'arms': 'manufacturing',
    'metallurgy': 'manufacturing',
    'ferrous': 'manufacturing',
    'steel': 'manufacturing',
    'metal': 'manufacturing',
    'aluminium': 'manufacturing',
    'ceramics': 'manufacturing',
    'ceramic': 'manufacturing',
    'porcelain': 'manufacturing',
    'pottery': 'manufacturing',
    'glass': 'manufacturing',
    'cement': 'manufacturing',
    'materials': 'manufacturing',
    'shipbuilding': 'manufacturing',
    'shipyard': 'manufacturing',
    'engine': 'manufacturing',
    'watchmaking': 'manufacturing',
    'watchmaker': 'manufacturing',
    'horology': 'manufacturing',
    'jewelry': 'manufacturing',
    'toy': 'manufacturing',
    'confectionery': 'manufacturing',
    'confection': 'manufacturing',
    'dairy': 'manufacturing',
    'tobacco': 'manufacturing',

    # --- Technology & Media -> Matches "Information" ---
    'telecommunications': 'information',
    'telecommunication': 'information',
    'internet': 'information',
    'software': 'information',
    'film': 'information',
    'filmmaking': 'information',
    'cinematography': 'information',
    'television': 'information',
    'video': 'information',
    'media': 'information',
    'publishing': 'information',
    'music': 'information',
    'phonographic': 'information',
    'animation': 'information',
    'broadcasting': 'information',
    'journalism': 'information',
    'news': 'information',
    'magazine': 'information',
    'syndication': 'information',
    'multimedia': 'information',
    'audiovisual': 'information',
    'podcast': 'information',
    'cinema': 'information',
    'graphics': 'information',
    'advertising': 'information',

    # --- Money -> Matches "Finance and Insurance" ---
    'bank': 'finance',
    'banking': 'finance',
    'financials': 'finance',
    'economics': 'finance',
    'investment': 'finance',
    'venture': 'finance',
    'equity': 'finance',
    'hedge': 'finance',
    'neobank': 'finance',
    'microfinance': 'finance',

    # --- Moving Things -> Matches "Transportation and Warehousing" ---
    'logistics': 'transportation',
    'transport': 'transportation',
    'freight': 'transportation',
    'rail': 'transportation',
    'railway': 'transportation',
    'shipping': 'transportation',
    'aviation': 'transportation',
    'mail': 'transportation',
    'postal': 'transportation',

    # --- Selling Things -> Matches "Retail Trade" ---
    'e-commerce': 'retail',
    'retail': 'retail',
    'store': 'retail',
    'shopping': 'retail',

    # --- Office, Science & Tech -> Matches "Professional, Scientific, and Technical Services" ---
    'architecture': 'technical',
    'architectural': 'technical',
    'engineering': 'technical',
    'research': 'technical',
    'consulting': 'technical',
    'design': 'technical',
    'robotics': 'technical',
    'ai': 'technical',
    'artificial': 'technical',
    'machine': 'technical',

    # --- Corporate & Admin -> Matches "Management of Companies and Enterprises" ---
    'commercial': 'management',
    'supplies': 'management',
    'business-to-business': 'management',
    'conglomerate': 'management',
    'holding': 'management',
    'corporate': 'management',

    # --- Health -> Matches "Health Care and Social Assistance" ---
    'healthcare': 'health',
    'medical': 'health',
    'pharmacy': 'health',

    # --- Utilities -> Matches "Utilities" ---
    'energy': 'utilities',
    'electricity': 'utilities',
    'hydroelectricity': 'utilities',
    'photovoltaics': 'utilities',
    'solar': 'utilities',
    'water': 'utilities',

    # --- Oil & Gas -> Matches "Mining, Quarrying, and Oil and Gas Extraction" ---
    'petroleum': 'extraction',

    # --- Arts & Fun -> Matches "Arts, Entertainment, and Recreation" ---
    'theatre': 'entertainment',
    'creative': 'entertainment',
    'gambling': 'entertainment',

    # --- Education -> Matches "Educational Services" ---
    'education': 'educational',
    'student': 'educational', 
    'battery': 'manufacturing',
    'machinery': 'manufacturing',
    'equipment': 'manufacturing',
    'pulp': 'manufacturing',
    'paper': 'manufacturing',
    'furniture': 'manufacturing',
    'musical': 'manufacturing',
    'amplifier': 'manufacturing',
    'boats': 'manufacturing',
    'watches': 'manufacturing',
    'clocks': 'manufacturing',
    'explosives': 'manufacturing',
    'meatpacking': 'manufacturing',
    'meat': 'manufacturing',
    'pasta': 'manufacturing',
    'bottling': 'manufacturing',
    'tyre': 'manufacturing',
    'components': 'manufacturing',
    'printing': 'manufacturing',

    # --- Information & Tech (New Items) ---
    'pornography': 'information',
    'comics': 'information',
    'game': 'information',
    'hosting': 'information',
    'computing': 'information',
    'tech': 'information',
    'technology': 'information',
    'it': 'information',
    'multimedia': 'information',
    'streaming': 'information',
    'publisher': 'information',
    'newspaper': 'information',

    # --- Retail & Wholesale (New Items) ---
    'drugstore': 'retail',
    'bookselling': 'retail',
    'dealership': 'retail',
    'shop': 'retail',
    'stationery': 'retail',
    'eyewear': 'retail',
    'goods': 'retail',
    'discretionary': 'retail',

    # --- Finance & Professional (New Items) ---
    'fintech': 'finance',
    'mortgage': 'finance',
    'loan': 'finance',
    'trading': 'finance',
    'market': 'finance',
    'analysis': 'technical',
    'automation': 'technical',
    'meteorology': 'technical',

    # --- Utilities & Energy (New Items) ---
    'power': 'utilities',
    'utility': 'utilities',
    'transmission': 'utilities',
    'coal': 'extraction',
    'fuel': 'extraction',

    # --- Transportation (New Items) ---
    'cruise': 'transportation',
    'aircraft': 'transportation',
    'tanker': 'transportation',
    'carsharing': 'transportation',

    # --- Other Services & Admin ---
    'staffing': 'management',
    'recruiting': 'management',
    'industrials': 'management',
    'hairdresser': '81',  # Matches "Other Services"
    'prison': 'administration',       # Matches "Public Administration"
    'nonprofit': '81' ,
    'light': 'manufacturing',
    'industrial': 'manufacturing',
    'production': 'manufacturing',
    'candy': 'manufacturing',
    'photo': 'manufacturing',
    'photography': 'manufacturing',

    # --- Information ---
    'digital': 'information',
    'motion': 'information',
    'picture': 'information',
    'show': 'information',

    # --- Retail & Wholesale ---
    'retailing': 'retail',
    'distribution': 'wholesale',

    # --- Agriculture ---
    'agribusiness': 'agriculture',

    # --- Finance ---
    'economy': 'finance',
    'tertiary': 'finance',

    # --- Administration (Updated per request) ---
    '81': 'administration',
    'prison': 'administration',
    'government': 'administration' ,
    'cannabidiol': 'manufacturing',  # Chemical/Pharmaceutical manufacturing
    'appliance': 'manufacturing',
    'bicycle': 'manufacturing',
    'cream': 'manufacturing',        # Matches "Ice cream manufacturing"
    
    # --- Information ---
    'sex': 'information',            # Niche media/entertainment usually falls here
    
    # --- Professional & Management ---
    'startup': 'technical',          # Most accelerators are Professional/Technical services
    'accelerator': 'technical',
    
    # --- Arts & Recreation ---
    'leisure': 'entertainment',
    'voice': 'information',
'ip': 'information',
'selling': 'retail',
'direct': 'retail',
'sports': 'entertainment',
'promoter': 'entertainment',
'racing': 'entertainment',
'auto': 'manufacturing',
'radio': 'information',
'communications': 'information',
'automobile': 'manufacturing',
'house-building': 'construction',
'alcohol': 'manufacturing',
'electronic': 'manufacturing',
'visual': 'manufacturing',
'display': 'manufacturing',
'art': 'entertainment',
'material': 'manufacturing',
'mobile': 'information',
'payment': 'finance',
'associated': 'management',
'brand': 'management',
'licensing': 'management',
'optics': 'manufacturing',
'military': 'administration',
'private': 'management',
'electrical': 'manufacturing',
'device': 'manufacturing',
'weather': 'technical',
'forecasting': 'technical',
'capital': 'finance',
'participation': 'management',
'product': 'manufacturing',
'packaging': 'manufacturing',
'truck': 'transportation',
'stop': 'accommodation',
'furnishings': 'manufacturing',
'export': 'wholesale',
'staples': 'retail',
'cash-in-transit': 'finance',
'fitness': 'entertainment',
'measurement': 'technical',
'technique': 'technical',
'auction': 'retail',
'commerce': 'retail',
'petroleum-gas': 'extraction',
'winemaking': 'manufacturing',
'potato': 'manufacturing',
'chip': 'manufacturing',
'security': 'technical',
'motorcycle': 'manufacturing',
'parts': 'manufacturing',
'accessories': 'manufacturing',
'vehicles': 'manufacturing',
'philately': 'retail',
'instruments': 'manufacturing',
'appliances': 'manufacturing',
'measuring': 'manufacturing',
'testing': 'manufacturing',
'navigation': 'manufacturing',
'skateboarding': 'entertainment',
'mattress': 'manufacturing',
'bed': 'manufacturing',
'base': 'manufacturing',
'pillow': 'manufacturing',
'cryonics': 'technical',
'motorsport': 'entertainment',
'bicycle-sharing': 'transportation',
'amusement': 'entertainment',
'ride': 'entertainment',
'filling': 'retail',
'goldsmithing': 'manufacturing',
'sport': 'entertainment',
'furnishing': 'manufacturing',
'bench': 'manufacturing',
'jeweler': 'manufacturing',
'comic': 'information',
'book': 'information',
'sound': 'information',
'effect': 'information',
'climbing': 'entertainment',
'space': 'transportation',
'ammunition': 'manufacturing',
'nanotechnology': 'technical',
'viniculture': 'agriculture',
'embedded': 'technical',
'system': 'technical',
'rehabilitation': 'health',
'computer-generated': 'information',
'imagery': 'information',
'fishery': 'agriculture',
'cloud': 'information',
'storage': 'information',
'family': 'finance',
'office': 'finance',
'file-hosting': 'information',
'optical': 'manufacturing',
'imaging': 'manufacturing',
'gardening': 'agriculture', 
'caroma': 'manufacturing',
'dvd': 'information',
'blu-ray': 'information',
'disc': 'information',
'laserdisc': 'information',
'post-production': 'information',
'recording': 'information',
'medium': 'information',
'computer': 'information',
'organization': 'administration',
'housing': 'finance',
'dating': 'information',
'marketing': 'technical',
'distilling': 'manufacturing',
'rectifying': 'manufacturing',
'blending': 'manufacturing',
'spirits': 'manufacturing',
'microdistillery': 'manufacturing',
'email': 'technical',
'kashrut': 'technical',
'stage': 'entertainment',
'lighting': 'manufacturing',
'microphone': 'manufacturing',
'intermodal': 'transportation',
'container': 'transportation',
'wine': 'manufacturing',
'talent': 'management',
'agent': 'management',
'wi-fi': 'information',
'chinese': 'accommodation',
'cuisine': 'accommodation',
'middleware': 'information',
'online': 'information',
'chocolatier': 'manufacturing',
'audio': 'manufacturing',
'signal': 'technical',
'processing': 'information',
'credit': 'finance',
'rating': 'finance',
'organ': 'manufacturing',
'builder': 'manufacturing',
'shoe': 'manufacturing',
'plumbing': 'construction',
'tour': 'transportation',
'operator': 'transportation',
'bijou': 'manufacturing',
'communication': 'information',
'videotelephony': 'information',
'genealogy': 'technical',
'hat': 'manufacturing',
'dredging': 'construction',
'ecology': 'technical',
'conservation': 'technical',
'vindication': 'technical',
'rights': 'technical',
'men': 'technical',
'heavy': 'manufacturing',
'random-access': 'manufacturing',
'memory': 'manufacturing',
'sōgō': 'wholesale',
'shōsha': 'wholesale',
'chemistry': 'technical',
'technology': 'information'
}

In [15]:
data['industry'] = data['industry'].apply(lambda word_list: [synonyms.get(word, word) for word in word_list]
)

In [16]:
toremove = {'and' , 'or' , 'the' , 'of' , 'in' , 'with' , '&', 'company' , 'industry',
 'to' , 'from' , 'by' , 'sector' , 'other' , 'except' ,'a' , 'services', 'service'} 

data['industry'] = data['industry'].apply(lambda x: [item.lower() for item in x if item.lower() not in toremove]) 

In [17]:
data['industry'] = data['industry'].apply(lambda x: ['finance' if item == 'financial' else item for item in x ]) 

In [18]:
data.head()

,company name,description,industry,original_industry
0,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",[accommodation],['hotel']
1,Goldman Sachs,American investment bank,"[finance, finance, finance, activities, insura...",['financial services' 'financial sector'\n 'fi...
2,Sberbank,Russian banking and financial services company,"[finance, russia, finance, finance, activities...",['banking in Russia' 'financial sector'\n 'fin...
3,Heineken,Dutch beer company,[manufacturing],['beverage industry']
4,Deutsche Bank,German global banking and financial services c...,"[finance, monetary, intermediation, finance, f...",['financial services' 'Other monetary intermed...


In [19]:
data.head()

,company name,description,industry,original_industry
0,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",[accommodation],['hotel']
1,Goldman Sachs,American investment bank,"[finance, finance, finance, activities, insura...",['financial services' 'financial sector'\n 'fi...
2,Sberbank,Russian banking and financial services company,"[finance, russia, finance, finance, activities...",['banking in Russia' 'financial sector'\n 'fin...
3,Heineken,Dutch beer company,[manufacturing],['beverage industry']
4,Deutsche Bank,German global banking and financial services c...,"[finance, monetary, intermediation, finance, f...",['financial services' 'Other monetary intermed...


In [20]:
def find_naics_code(sectors):
    row = set(sectors)

    for naics in title_code_matching:
        naics_titles = naics.lower().replace(",", "").replace(")", "").replace("(", "")
        naics_titles = set(naics_titles.split(" ") ) 
        overlap = row & naics_titles 
        if len(overlap) > 0:
            print(f'{row} matches to {naics}') 
            return naics 

    return None 


data['naics_title'] = data['industry'].apply(find_naics_code)

{'accommodation'} matches to Accommodation and Food Services
{'insurance', 'pension', 'finance', 'funding', 'activities'} matches to Finance and Insurance
{'insurance', 'pension', 'finance', 'funding', 'russia', 'activities'} matches to Finance and Insurance
{'manufacturing'} matches to Manufacturing
{'insurance', 'pension', 'finance', 'funding', 'intermediation', 'monetary', 'activities'} matches to Finance and Insurance
{'manufacturing', 'food'} matches to Manufacturing
{'manufacturing'} matches to Manufacturing
{'finance'} matches to Finance and Insurance
{'food'} matches to Accommodation and Food Services
{'finance', 'insurance'} matches to Finance and Insurance
{'phone', 'information'} matches to Information
{'finance'} matches to Finance and Insurance
{'manufacturing'} matches to Manufacturing
{'finance'} matches to Finance and Insurance
{'insurance', 'pension', 'finance', 'funding', 'activities'} matches to Finance and Insurance
{'finance'} matches to Finance and Insurance
{'fin

In [21]:
data['naics_title'].isna().sum()

np.int64(1)

In [22]:
data.head()

,company name,description,industry,original_industry,naics_title
0,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",[accommodation],['hotel'],Accommodation and Food Services
1,Goldman Sachs,American investment bank,"[finance, finance, finance, activities, insura...",['financial services' 'financial sector'\n 'fi...,Finance and Insurance
2,Sberbank,Russian banking and financial services company,"[finance, russia, finance, finance, activities...",['banking in Russia' 'financial sector'\n 'fin...,Finance and Insurance
3,Heineken,Dutch beer company,[manufacturing],['beverage industry'],Manufacturing
4,Deutsche Bank,German global banking and financial services c...,"[finance, monetary, intermediation, finance, f...",['financial services' 'Other monetary intermed...,Finance and Insurance


In [23]:
missing_industry = data.loc[data['naics_title'].isna() , 'industry'] 
missing_industry = missing_industry.to_list() 
missing_industry

[['81']]

In [24]:
data.loc[data['company name'] == 'Toni & Guy' , 'naics_title'] = 'Other Services (except Public Administration)' 

In [25]:
missing_sector = data[data['naics_title'].isna()]
missing_sector

,company name,description,industry,original_industry,naics_title


In [26]:
data.head(10)

,company name,description,industry,original_industry,naics_title
0,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",[accommodation],['hotel'],Accommodation and Food Services
1,Goldman Sachs,American investment bank,"[finance, finance, finance, activities, insura...",['financial services' 'financial sector'\n 'fi...,Finance and Insurance
2,Sberbank,Russian banking and financial services company,"[finance, russia, finance, finance, activities...",['banking in Russia' 'financial sector'\n 'fin...,Finance and Insurance
3,Heineken,Dutch beer company,[manufacturing],['beverage industry'],Manufacturing
4,Deutsche Bank,German global banking and financial services c...,"[finance, monetary, intermediation, finance, f...",['financial services' 'Other monetary intermed...,Finance and Insurance
5,Carlsberg Group,Danish brewery group,"[food, manufacturing]",['food industry' 'beverage industry'],Manufacturing
6,Beck's,German brewery,[manufacturing],['brewing'],Manufacturing
7,Kasikornbank,Financial institution in Thailand,"[finance, finance]",['tertiary sector of the economy'],Finance and Insurance
8,A. Le Coq,Estonian brewery,[food],['food industry'],Accommodation and Food Services
9,HSBC,British multinational bank,"[finance, insurance, finance, finance, finance]",['financial services' 'insurance' 'financial s...,Finance and Insurance


In [27]:
SECTOR_MERGE = {
    '31': '31-33', '32': '31-33', '33': '31-33',
    '44': '44-45', '45': '44-45',
    '48': '48-49', '49': '48-49',
}

naics_code_df = pd.read_csv('../ExioNAICS.csv')[['NAICS_2 Title', 'NAICS_2 Code']]
naics_code_df = naics_code_df.rename(columns={'NAICS_2 Title': 'naics_title', 'NAICS_2 Code': 'naics2_code'})

naics_code_df['naics2_code'] = naics_code_df['naics2_code'].astype(str)
naics_code_df['naics2_code'] = naics_code_df['naics2_code'].map(SECTOR_MERGE).fillna(naics_code_df['naics2_code'])

naics_code_df = naics_code_df.drop_duplicates(subset='naics_title').reset_index(drop=True)

df = pd.merge(data, naics_code_df, on='naics_title', how='left')

print(f"Rows in data: {len(data)}")
print(f"Rows in df:   {len(df)}  (should match)")
print(f"\nUnique sector codes in merged df:")
print(sorted(df['naics2_code'].unique()))
print(f"\nMissing codes: {df['naics2_code'].isna().sum()}")

df.head()

Rows in data: 77553
Rows in df:   77553  (should match)

Unique sector codes in merged df:
['11', '21', '22', '23', '31-33', '42', '44-45', '48-49', '51', '52', '53', '54', '55', '56', '61', '62', '71', '72', '81']

Missing codes: 0


,company name,description,industry,original_industry,naics_title,naics2_code
0,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",[accommodation],['hotel'],Accommodation and Food Services,72
1,Goldman Sachs,American investment bank,"[finance, finance, finance, activities, insura...",['financial services' 'financial sector'\n 'fi...,Finance and Insurance,52
2,Sberbank,Russian banking and financial services company,"[finance, russia, finance, finance, activities...",['banking in Russia' 'financial sector'\n 'fin...,Finance and Insurance,52
3,Heineken,Dutch beer company,[manufacturing],['beverage industry'],Manufacturing,31-33
4,Deutsche Bank,German global banking and financial services c...,"[finance, monetary, intermediation, finance, f...",['financial services' 'Other monetary intermed...,Finance and Insurance,52


In [28]:
df = df[['company name' , 'description', 'naics_title', 'naics2_code']]


In [29]:
df = df.rename(columns = {'naics_title': 'naics2_title'})
df.head()

,company name,description,naics2_title,naics2_code
0,Hotel Sacher,"building in Innere Stadt, Vienna, Austria",Accommodation and Food Services,72
1,Goldman Sachs,American investment bank,Finance and Insurance,52
2,Sberbank,Russian banking and financial services company,Finance and Insurance,52
3,Heineken,Dutch beer company,Manufacturing,31-33
4,Deutsche Bank,German global banking and financial services c...,Finance and Insurance,52


In [30]:
print(f"Before cleanup: {df.shape}")

df = df.dropna(subset=['description']).reset_index(drop=True)
print(f"After dropping null descriptions: {df.shape}")

df = df.drop_duplicates(subset=['company name', 'description']).reset_index(drop=True)
print(f"After dropping (name, desc) duplicates: {df.shape}")

df = df.drop_duplicates(subset=['description']).reset_index(drop=True)
print(f"After dropping duplicate descriptions:  {df.shape}")

print(f"\nNulls per column:")
print(df.isna().sum().to_string())

print(f"\nClass distribution (naics2_code):")
print(df['naics2_code'].value_counts().sort_index().to_string())
print(f"\nNum unique sectors: {df['naics2_code'].nunique()}")

In [31]:
df.to_csv('ood_dataset_official.csv', index=False)